In [1]:
!apt-get update -y && apt-get install -y poppler-utils
!pip install -q "pandas<3.0.0" pdf2image pillow tqdm
!pip install -q -U pydantic pydantic-core google-genai

Get:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:2 https://cli.github.com/packages stable InRelease [3,917 B]               
Get:3 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [101 kB]
Get:4 https://cli.github.com/packages stable/main amd64 Packages [355 B]       
Get:5 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]      
Hit:6 http://archive.ubuntu.com/ubuntu jammy InRelease                         
Get:7 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]        
Get:8 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]           
Get:9 https://r2u.stat.illinois.edu/ubuntu jammy/main all Packages [10.6 MB]   
Get:10 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Get:11 http://security.ubuntu.com/ubuntu jammy-security/universe amd64 Packages [1,313 kB]
Hit:12 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease   
Get:13 http:/

In [2]:
import sys
import subprocess
import pdf2image
import PIL
from google import genai
from pdf2image import convert_from_path
import os
from tqdm.auto import tqdm
import time
import json
from PIL import Image
from tqdm.notebook import tqdm
from google.genai import types

In [3]:
print(f"Python Version: {sys.version.split()[0]}")
print(f"Pillow Version: {PIL.__version__}")

# Verify poppler binary (pdftoppm)
poppler_check = subprocess.run(["pdftoppm", "-v"], capture_output=True, text=True)
if poppler_check.returncode == 0:
    print("✅ SUCCESS: Poppler-utils (pdftoppm) is installed and ready in PATH!")
else:
    print("❌ ERROR: Poppler-utils installation failed.")

Python Version: 3.12.13
Pillow Version: 11.3.0
✅ SUCCESS: Poppler-utils (pdftoppm) is installed and ready in PATH!


In [ ]:
try:
    API_KEY = ""
    print("✅ GEMINI_API_KEY retrieved!")
except Exception:
    API_KEY = "YOUR_GEMINI_API_KEY_HERE"
    print("⚠️ Kaggle Secrets not detected. Using manual string assignment.")

client = genai.Client(api_key=API_KEY)

✅ GEMINI_API_KEY retrieved!


In [5]:
# Update this path to where your input PDF is located in Kaggle
PDF_INPUT_PATH = "/kaggle/input/datasets/gawwyzeghost/hsc-math-question-bank/Phy2 quebank.pdf" 
TEMP_IMAGE_DIR = "/kaggle/working/pdf_page_images"

def preprocess_pdf_to_images(pdf_path, output_dir, dpi=300):
    os.makedirs(output_dir, exist_ok=True)
    
    if not os.path.exists(pdf_path):
        print(f"❌ File '{pdf_path}' not found. Please upload a PDF or update the dataset path.")
        return []
        
    print(f"Converting '{pdf_path}' to images at {dpi} DPI...")
    images = convert_from_path(pdf_path, dpi=dpi)
    
    page_records = []
    print(f"Saving {len(images)} pages as images...")
    
    for index, img in enumerate(tqdm(images, desc="Saving Pages", unit="page")):
        page_num = index + 1
        page_path = os.path.join(output_dir, f"page_{page_num}.png")
        img.save(page_path, "PNG")
        page_records.append((page_num, page_path))
        
    print(f"✅ Converted {len(images)} PDF page(s) into high-res images.")
    return page_records

page_list = preprocess_pdf_to_images(PDF_INPUT_PATH, TEMP_IMAGE_DIR)

Converting '/kaggle/input/datasets/gawwyzeghost/hsc-math-question-bank/Phy2 quebank.pdf' to images at 300 DPI...
Saving 103 pages as images...


Saving Pages:   0%|          | 0/103 [00:00<?, ?page/s]

✅ Converted 103 PDF page(s) into high-res images.


In [6]:
import time
import re
from PIL import Image
from tqdm.notebook import tqdm
from google.genai import types
from google.genai.errors import APIError

SYSTEM_PROMPT = """
You are an expert Bengali HSC exam digitizer for Higher Math, Physics, and Chemistry.
Extract all questions (MCQ and Written/CQ) from this page image into structured JSON format.

RULES:
1. Output ONLY a valid JSON array of objects. Do NOT wrap inside markdown code block wrappers.
2. Keep all Bengali text strictly in Unicode characters.
3. Convert all mathematical equations, chemical formulas, and physical units into standard LaTeX inside single dollar signs ($...$).
   - Example Math: $\\int_0^{\\pi/2} \\sin^2(x) dx$
   - Example Physics: $v = u + at$, $\\vec{F} = m\\vec{a}$
   - Example Chemistry: $\\text{H}_2\\text{SO}_4 + 2\\text{NaOH} \\rightarrow \\text{Na}_2\\text{SO}_4 + 2\\text{H}_2\\text{O}$
4. Extract exam tags if present (e.g., [BUET '18-19], [Dhaka Board '22]) into 'source_tag'.
5. MUST ALWAYS provide 'options' as a dictionary object. 
   - For MCQs: Dynamically extract ALL options present on the page as key-value pairs (e.g., {"a": "...", "b": "...", "c": "...", "d": "...", "e": "..."}). Include as many keys as there are choices printed on the page.
   - For Written/CQ questions: Set 'options' to an empty object {}.
6. If a part of the question is cut off from a page and only one is available or partially available, simply fill in the empty fields with null.

JSON Schema per question:
[
  {
    "question_number": "01",
    "question_type": "MCQ or Written",
    "subject": "Higher Math or Physics or Chemistry",
    "source_tag": "string or null",
    "question_text": "string with LaTeX",
    "options": {
      "a": "string or null"
    },
    "answer": "string or null",
    "solution": "string or null",
    "page_number": 15
  }
]

Examples:

{
    "question_number": "14",
    "question_type": "Written",
    "subject": "Chemistry",
    "source_tag": "[BUTex'09-10]",
    "question_text": "কোন কোয়ান্টাম সংখ্যা দ্বারা $\\bar{e}$ এর শক্তিস্তরের আকৃতি বোঝা যায়?",
    "options": null,
    "answer": null,
    "solution": "সমাধান: সহকারী কোয়ান্টাম সংখ্যা দ্বারা।",
    "page_number": 6
}

{
    "question_number": "01",
    "question_type": "MCQ",
    "subject": "Chemistry",
    "source_tag": "KUET'18-19",
    "question_text": "একটি মৌলের ইলেকট্রন বিন্যাস হলো $1s^2 2s^2 2p^6 3s^2 3p^5$। মৌলটির $6.24078 \\times 10^{-23}\\text{ g}$ এ কতটি ইলেকট্রন আছে?",
    "options": {
      "a": "$1$",
      "b": "$1.02391 \\times 10^{23}$",
      "c": "$1.64067 \\times 10^{47}$",
      "d": "$18$ (অথবা $639$)"
    },
    "answer": "d",
    "solution": "$1\\text{ mol Cl}$ এ $e^- = 17 \\times 6.023 \\times 10^{23}$ টি $e^-$\\n$35.5\\text{ g Cl}$ এ $e^- = 17 \\times 6.023 \\times 10^{23}$\\n$\\therefore 6.24078 \\times 10^{-23}\\text{ g}$ এ $e^- = \\frac{17 \\times 6.023 \\times 10^{23} \\times 6.24078 \\times 10^{-23}}{35.5} = 18$ টি $e^-$",
    "page_number": 8
}

{
    "question_number": "54",
    "question_type": "MCQ",
    "subject": "Chemistry",
    "source_tag": "[KUET'13-14]",
    "question_text": "একটি পরমাণুর বহিঃস্তরের ইলেকট্রনীয় কাঠামো $...3d^5 4s^1$। পর্যায় সারণীতে মৌলটির অবস্থান কোনটি?",
    "options": {
      "a": "Group 1A, Period 3",
      "b": "Group 1B, Period 4",
      "c": "Group 1VA, Period 4",
      "d": "Group 1VA, Period 3",
      "e": "Group 6B, Period 4"
    },
    "answer": "e",
    "solution": null,
    "page_number": 14
}
"""

def process_extracted_pages(
    page_records, 
    txt_output_path="/kaggle/working/extracted_questions.txt", 
    start_page=None, 
    end_page=None, 
    model_name="gemini-3.1-flash-lite"
):
    if not page_records:
        print("No page images found to process.")
        return

    # Range selection (1-based, inclusive)
    if start_page is not None or end_page is not None:
        start_idx = (start_page - 1) if start_page is not None and start_page > 0 else 0
        end_idx = end_page if end_page is not None else len(page_records)
        target_records = page_records[start_idx:end_idx]
        print(f"🎯 RANGE SELECTION: Processing pages {start_idx + 1} to {min(end_idx, len(page_records))} (inclusive) out of {len(page_records)} pages using '{model_name}'.\n")
    else:
        target_records = page_records
        print(f"🚀 FULL BATCH MODE: Processing all {len(target_records)} pages using '{model_name}'.\n")

    progress_bar = tqdm(target_records, desc="Extracting PDF Pages", unit="page")

    # Open target text file in write mode
    with open(txt_output_path, "w", encoding="utf-8") as f:
        for page_num, img_path in progress_bar:
            progress_bar.set_postfix({"Current Page": page_num})
            img = Image.open(img_path)
            
            # Explicit dynamic force instruction overriding any printed text on image
            dynamic_prompt = (
                f"STRICT INSTRUCTION: For ALL items extracted from this image, you MUST set "
                f"'page_number': {page_num}. Ignore any printed page numbers on the document itself.\n\n"
                f"{SYSTEM_PROMPT}"
            )
            
            success = False
            retries = 0
            max_retries = 3
            
            while not success and retries < max_retries:
                retries += 1
                try:
                    response = client.models.generate_content(
                        model=model_name,
                        contents=[img, dynamic_prompt],
                        config=types.GenerateContentConfig(
                            temperature=0.1,
                            response_mime_type="application/json"
                        )
                    )
                    
                    raw_text = response.text.strip()
                    
                    # Write raw response directly to disk
                    f.write(raw_text + "\n\n")
                    f.flush()
                    
                    success = True
                    time.sleep(12)  # Rate limit safety delay
                    
                except APIError as e:
                    err_msg = str(e)
                    
                    # Catch Rate Limit (429) OR Server High Demand (503 / 500)
                    if any(code in err_msg for code in ["429", "503", "500", "RESOURCE_EXHAUSTED", "UNAVAILABLE"]):
                        match = re.search(r"retry in (\d+(\.\d+)?)s", err_msg)
                        wait_time = float(match.group(1)) + 2 if match else 15
                        progress_bar.write(f"⏳ Server Busy / Quota hit on Page {page_num}. Pausing {int(wait_time)}s (Attempt {retries}/{max_retries})...")
                        time.sleep(wait_time)
                    else:
                        progress_bar.write(f"⚠️ API Error on Page {page_num}: {e}")
                        break
                        
                except Exception as e:
                    progress_bar.write(f"⚠️ Unexpected error on Page {page_num}: {e}")
                    break

    print(f"\n✅ Finished processing!")
    print(f"Raw response saved to: {txt_output_path}")

print("Loaded function")

Loaded function


In [8]:
extracted_data = process_extracted_pages(page_list, start_page=16, end_page=18)

🎯 RANGE SELECTION: Processing pages 2 to 103 (inclusive) out of 103 pages using 'gemini-3.1-flash-lite'.



Extracting PDF Pages:   0%|          | 0/102 [00:00<?, ?page/s]


✅ Finished processing!
Raw response saved to: /kaggle/working/extracted_questions.txt
